In [2]:
def process_bone_data():
    input_file = r"C:\Users\user\Desktop\raw_data.txt"
    output_file = r"C:\Users\user\Desktop\raw_data3.txt"
    
    try:
        # 讀取原始檔案
        with open(input_file, 'r', encoding='cp932') as f:
            lines = f.readlines()
        
        # 處理數據
        filtered_lines = []
        current_bone_lines = []
        skip_current_bone = False
        
        for line in lines:
            line = line.strip()
            
            # 如果是新的骨骼記錄開始
            if line.startswith("FRAME:"):
                # 先處理前一個骨骼的記錄
                if current_bone_lines and not skip_current_bone:
                    filtered_lines.extend(current_bone_lines)
                
                # 重置狀態
                current_bone_lines = [line + '\n']
                skip_current_bone = False
                
            elif line.startswith("FINAL_POS:"):
                # 檢查 FINAL_POS 是否為 0.000,0.000,0.000
                pos_values = line.split(':')[1].strip()
                if pos_values == "0.000,0.000,0.000":
                    skip_current_bone = True
                    print(f"刪除骨骼記錄 - FINAL_POS: {pos_values}")
                
                current_bone_lines.append(line + '\n')
                
            else:
                # 其他行直接加入當前骨骼記錄
                if line:  # 避免空行
                    current_bone_lines.append(line + '\n')
        
        # 處理最後一個骨骼記錄
        if current_bone_lines and not skip_current_bone:
            filtered_lines.extend(current_bone_lines)
        
        # 寫入結果檔案
        with open(output_file, 'w', encoding='utf-8') as f:
            f.writelines(filtered_lines)
        
        print(f"處理完成！")
        print(f"原始檔案: {input_file}")
        print(f"輸出檔案: {output_file}")
        print(f"已刪除 FINAL_POS 為 0.000,0.000,0.000 的骨骼記錄")
        
    except FileNotFoundError:
        print(f"錯誤：找不到檔案 {input_file}")
    except Exception as e:
        print(f"處理過程中發生錯誤: {e}")

# 執行腳本
if __name__ == "__main__":
    process_bone_data()

刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FINAL_POS: 0.000,0.000,0.000
刪除骨骼記錄 - FIN

In [13]:
import fnmatch

def process_bone_data():
    input_file = r"C:\Users\user\Desktop\raw_data.txt"
    output_file = r"C:\Users\user\Desktop\raw_data3.txt"
    
    # 設定白名單和黑名單
    whitelist = ["左足ＩＫ", "左足D", "左ひざD", "左足首D"]  # 白名單：只保留匹配的骨骼
    whitelist = ["*目*"]  # 白名單：只保留匹配的骨骼
    blacklist = ["*Hair*", "*Skirt*", "*bone*"]  # 黑名單：剔除匹配的骨骼
    
    def is_near_zero(value, tolerance=0.001):
        """檢查數值是否接近零（在誤差範圍內）"""
        return abs(float(value)) <= tolerance
    
    def is_near_one(value, tolerance=0.001):
        """檢查數值是否接近一（在誤差範圍內）"""
        return abs(float(value) - 1.0) <= tolerance
    
    def check_final_pos_zero(pos_line):
        """檢查 FINAL_POS 是否為 0.000,0.000,0.000（允許誤差）"""
        try:
            pos_values = pos_line.split(':')[1].strip()
            x, y, z = pos_values.split(',')
            return (is_near_zero(x) and is_near_zero(y) and is_near_zero(z))
        except:
            return False
    
    def check_final_rot_identity(rot_line):
        """檢查 FINAL_ROT 是否為 0.000,0.000,0.000,1.000（允許誤差）"""
        try:
            rot_values = rot_line.split(':')[1].strip()
            x, y, z, w = rot_values.split(',')
            return (is_near_zero(x) and is_near_zero(y) and is_near_zero(z) and is_near_one(w))
        except:
            return False
    
    def match_pattern_list(name, pattern_list):
        """檢查名稱是否匹配模式列表中的任一項（支持萬用字符）"""
        if not pattern_list:
            return False
        
        for pattern in pattern_list:
            if fnmatch.fnmatch(name, pattern):
                return True
        return False
    
    def get_bone_name(frame_line):
        """從 FRAME 行中提取骨骼名稱"""
        try:
            # FRAME:0,BONE:223,NAME:HairB7
            parts = frame_line.split(',')
            for part in parts:
                if part.startswith('NAME:'):
                    return part.split('NAME:')[1].strip()
            return ""
        except:
            return ""
    
    def should_process_bone(bone_name):
        """根據白名單和黑名單規則判斷是否應該處理此骨骼"""
        # 第一步：如果白名單不為空，檢查是否在白名單中
        if whitelist:
            if not match_pattern_list(bone_name, whitelist):
                return False, "不在白名單中"
        
        # 第二步：如果黑名單不為空，檢查是否在黑名單中
        if blacklist:
            if match_pattern_list(bone_name, blacklist):
                return False, "在黑名單中"
        
        return True, "通過名單檢查"
    
    try:
        # 讀取原始檔案
        with open(input_file, 'r', encoding='cp932') as f:
            lines = f.readlines()
        
        # 處理數據
        filtered_lines = []
        current_bone_lines = []
        skip_current_bone = False
        final_pos_zero = False
        final_rot_identity = False
        current_bone_name = ""
        skip_reason = ""
        
        for line in lines:
            stripped_line = line.strip()
            
            # 如果是新的骨骼記錄開始
            if stripped_line.startswith("FRAME:"):
                # 先處理前一個骨骼的記錄
                if current_bone_lines:
                    if skip_current_bone:
                        print(f"刪除骨骼記錄 - {current_bone_name}: {skip_reason}")
                    else:
                        filtered_lines.extend(current_bone_lines)
                
                # 重置狀態並檢查新骨骼
                current_bone_name = get_bone_name(stripped_line)
                should_keep, reason = should_process_bone(current_bone_name)
                
                current_bone_lines = [line]
                skip_current_bone = not should_keep
                skip_reason = reason
                final_pos_zero = False
                final_rot_identity = False
                
            elif stripped_line.startswith("FINAL_POS:"):
                # 檢查 FINAL_POS 是否為零向量（允許誤差）
                final_pos_zero = check_final_pos_zero(stripped_line)
                current_bone_lines.append(line)
                
            elif stripped_line.startswith("FINAL_ROT:"):
                # 檢查 FINAL_ROT 是否為單位四元數（允許誤差）
                final_rot_identity = check_final_rot_identity(stripped_line)
                current_bone_lines.append(line)
                
                # 當處理完 FINAL_ROT 後，如果骨骼通過名單檢查，再檢查數值條件
                if not skip_current_bone and final_pos_zero and final_rot_identity:
                    skip_current_bone = True
                    skip_reason = "FINAL_POS 接近 (0,0,0) 且 FINAL_ROT 接近 (0,0,0,1)"
                
            else:
                # 其他行（包括空行）直接加入當前骨骼記錄
                current_bone_lines.append(line)
        
        # 處理最後一個骨骼記錄
        if current_bone_lines:
            if skip_current_bone:
                print(f"刪除骨骼記錄 - {current_bone_name}: {skip_reason}")
            else:
                filtered_lines.extend(current_bone_lines)
        
        # 寫入結果檔案
        with open(output_file, 'w', encoding='utf-8') as f:
            f.writelines(filtered_lines)
        
        print(f"處理完成！")
        print(f"原始檔案: {input_file}")
        print(f"輸出檔案: {output_file}")
        print(f"白名單: {whitelist if whitelist else '未設定'}")
        print(f"黑名單: {blacklist if blacklist else '未設定'}")
        print(f"允許誤差範圍: ±0.001")
        
    except FileNotFoundError:
        print(f"錯誤：找不到檔案 {input_file}")
    except Exception as e:
        print(f"處理過程中發生錯誤: {e}")

# 執行腳本
if __name__ == "__main__":
    process_bone_data()

刪除骨骼記錄 - 操作中心: 不在白名單中
刪除骨骼記錄 - 全ての親: 不在白名單中
刪除骨骼記錄 - センター: 不在白名單中
刪除骨骼記錄 - グルーブ: 不在白名單中
刪除骨骼記錄 - 腰: 不在白名單中
刪除骨骼記錄 - 下半身: 不在白名單中
刪除骨骼記錄 - 上半身: 不在白名單中
刪除骨骼記錄 - 上半身2: 不在白名單中
刪除骨骼記錄 - 首: 不在白名單中
刪除骨骼記錄 - 頭: 不在白名單中
刪除骨骼記錄 - 両目: FINAL_POS 接近 (0,0,0) 且 FINAL_ROT 接近 (0,0,0,1)
刪除骨骼記錄 - 腰キャンセル左: 不在白名單中
刪除骨骼記錄 - 左足: 不在白名單中
刪除骨骼記錄 - 左ひざ: 不在白名單中
刪除骨骼記錄 - 左足首: 不在白名單中
刪除骨骼記錄 - 左足首先: 不在白名單中
刪除骨骼記錄 - 腰キャンセル右: 不在白名單中
刪除骨骼記錄 - 右足: 不在白名單中
刪除骨骼記錄 - 右ひざ: 不在白名單中
刪除骨骼記錄 - 右足首: 不在白名單中
刪除骨骼記錄 - 右足首先: 不在白名單中
刪除骨骼記錄 - 左足IK親: 不在白名單中
刪除骨骼記錄 - 右足IK親: 不在白名單中
刪除骨骼記錄 - 左肩P: 不在白名單中
刪除骨骼記錄 - 左肩: 不在白名單中
刪除骨骼記錄 - 左肩C: 不在白名單中
刪除骨骼記錄 - 左腕: 不在白名單中
刪除骨骼記錄 - 左腕捩: 不在白名單中
刪除骨骼記錄 - 左腕捩1: 不在白名單中
刪除骨骼記錄 - 左腕捩2: 不在白名單中
刪除骨骼記錄 - 左腕捩3: 不在白名單中
刪除骨骼記錄 - 左ひじ: 不在白名單中
刪除骨骼記錄 - 左手捩: 不在白名單中
刪除骨骼記錄 - 左手捩1: 不在白名單中
刪除骨骼記錄 - 左手捩2: 不在白名單中
刪除骨骼記錄 - 左手捩3: 不在白名單中
刪除骨骼記錄 - 左手首: 不在白名單中
刪除骨骼記錄 - 左ダミー: 不在白名單中
刪除骨骼記錄 - 右肩P: 不在白名單中
刪除骨骼記錄 - 右肩: 不在白名單中
刪除骨骼記錄 - 右肩C: 不在白名單中
刪除骨骼記錄 - 右腕: 不在白名單中
刪除骨骼記錄 - 右腕捩: 不在白名單中
刪除骨骼記錄 - 右腕捩1: 不在白名單中
刪除骨骼記錄 - 右腕捩2: 不在白名單中
刪

In [17]:
import re
import math

# 全局容差參數 - 可以在這裡調整容差範圍
TOLERANCE = 0.001

def read_bone_data(file_path):
    """讀取骨骼數據文件"""
    try:
        with open(file_path, 'r', encoding='cp932') as file:
            content = file.read()
        return content
    except Exception as e:
        print(f"讀取文件時發生錯誤: {e}")
        return None

def parse_vector3(line, prefix):
    """解析3D向量數據"""
    pattern = f"{prefix}:([^,]+),([^,]+),([^,\\s]+)"
    match = re.search(pattern, line)
    if match:
        return [float(match.group(1)), float(match.group(2)), float(match.group(3))]
    return None

def parse_quaternion(line, prefix):
    """解析四元數數據"""
    pattern = f"{prefix}:([^,]+),([^,]+),([^,]+),([^,\\s]+)"
    match = re.search(pattern, line)
    if match:
        return [float(match.group(1)), float(match.group(2)), 
                float(match.group(3)), float(match.group(4))]
    return None

def calculate_max_difference(vec1, vec2):
    """計算兩個向量的最大差異值"""
    if vec1 is None or vec2 is None:
        return 0
    
    differences = [abs(a - b) for a, b in zip(vec1, vec2)]
    return max(differences)

def vectors_differ(vec1, vec2, tolerance):
    """檢查兩個向量是否在容差範圍內不同"""
    max_diff = calculate_max_difference(vec1, vec2)
    return max_diff > tolerance

def analyze_bone_data(file_path):
    """分析骨骼數據並找出差異"""
    content = read_bone_data(file_path)
    if content is None:
        return
    
    # 分割成各個骨骼的調試區塊
    bone_blocks = re.split(r'====== DEBUGGING', content)[1:]  # 跳過第一個空白部分
    
    # 使用字典來追蹤每個骨骼的最大差異
    pos_bone_data = {}
    rot_bone_data = {}
    
    for block in bone_blocks:
        lines = block.split('\n')
        
        # 提取骨骼名稱
        bone_name_match = re.search(r"'([^']+)'", lines[0])
        if not bone_name_match:
            continue
        bone_name = bone_name_match.group(1)
        
        vmd_pos = None
        final_pos = None
        vmd_rot = None
        final_rot = None
        
        # 解析每一行數據
        for line in lines:
            if line.startswith('VMD_POS:'):
                vmd_pos = parse_vector3(line, 'VMD_POS')
            elif line.startswith('FINAL_POS:'):
                final_pos = parse_vector3(line, 'FINAL_POS')
            elif line.startswith('VMD_ROT:'):
                vmd_rot = parse_quaternion(line, 'VMD_ROT')
            elif line.startswith('FINAL_ROT:'):
                final_rot = parse_quaternion(line, 'FINAL_ROT')
        
        # 檢查位置差異
        if vectors_differ(vmd_pos, final_pos, TOLERANCE):
            max_pos_diff = calculate_max_difference(vmd_pos, final_pos)
            
            # 如果骨骼名稱已存在，比較並保留最大差異的數據
            if bone_name not in pos_bone_data or max_pos_diff > pos_bone_data[bone_name]['max_diff']:
                pos_bone_data[bone_name] = {
                    'vmd_pos': vmd_pos,
                    'final_pos': final_pos,
                    'max_diff': max_pos_diff,
                    'all_diffs': [abs(a - b) for a, b in zip(vmd_pos, final_pos)] if vmd_pos and final_pos else []
                }
        
        # 檢查旋轉差異
        if vectors_differ(vmd_rot, final_rot, TOLERANCE):
            max_rot_diff = calculate_max_difference(vmd_rot, final_rot)
            
            # 如果骨骼名稱已存在，比較並保留最大差異的數據
            if bone_name not in rot_bone_data or max_rot_diff > rot_bone_data[bone_name]['max_diff']:
                rot_bone_data[bone_name] = {
                    'vmd_rot': vmd_rot,
                    'final_rot': final_rot,
                    'max_diff': max_rot_diff,
                    'all_diffs': [abs(a - b) for a, b in zip(vmd_rot, final_rot)] if vmd_rot and final_rot else []
                }
    
    # 輸出結果
    print("=== 位置差異分析結果 ===")
    if pos_bone_data:
        print(f"找到 {len(pos_bone_data)} 個位置不同的骨骼：")
        # 按最大差異排序
        sorted_pos_bones = sorted(pos_bone_data.items(), key=lambda x: x[1]['max_diff'], reverse=True)
        
        for bone_name, data in sorted_pos_bones:
            print(f"骨骼名稱: {bone_name}")
            print(f"  VMD_POS: {data['vmd_pos']}")
            print(f"  FINAL_POS: {data['final_pos']}")
            print(f"  各軸差異: {data['all_diffs']}")
            print(f"  最大差異: {data['max_diff']:.6f}")
            print()
    else:
        print("沒有找到位置差異超過容差範圍的骨骼")
    
    print("\n=== 旋轉差異分析結果 ===")
    if rot_bone_data:
        print(f"找到 {len(rot_bone_data)} 個旋轉不同的骨骼：")
        # 按最大差異排序
        sorted_rot_bones = sorted(rot_bone_data.items(), key=lambda x: x[1]['max_diff'], reverse=True)
        
        for bone_name, data in sorted_rot_bones:
            print(f"骨骼名稱: {bone_name}")
            print(f"  VMD_ROT: {data['vmd_rot']}")
            print(f"  FINAL_ROT: {data['final_rot']}")
            print(f"  各軸差異: {data['all_diffs']}")
            print(f"  最大差異: {data['max_diff']:.6f}")
            print()
    else:
        print("沒有找到旋轉差異超過容差範圍的骨骼")
    
    return pos_bone_data, rot_bone_data

if __name__ == "__main__":
    # 可以在這裡修改容差值
    print(f"當前容差設定: ±{TOLERANCE}")
    print("如要修改容差，請編輯腳本頂部的 TOLERANCE 變數")
    
    file_path = r"C:\Users\user\Desktop\raw_data.txt"
    
    print(f"\n開始分析骨骼數據...")
    print(f"文件路徑: {file_path}")
    print("="*60)
    
    pos_diff, rot_diff = analyze_bone_data(file_path)
    
    print("\n" + "="*60)
    print("分析完成！")
    print(f"位置差異骨骼數量: {len(pos_diff) if pos_diff else 0}")
    print(f"旋轉差異骨骼數量: {len(rot_diff) if rot_diff else 0}")

當前容差設定: ±0.001
如要修改容差，請編輯腳本頂部的 TOLERANCE 變數

開始分析骨骼數據...
文件路徑: C:\Users\user\Desktop\raw_data.txt
=== 位置差異分析結果 ===
找到 116 個位置不同的骨骼：
骨骼名稱: HairA15
  VMD_POS: [0.534, 0.552, 1.007]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.534, 0.552, 1.007]
  最大差異: 1.007000

骨骼名稱: bone0378_Ches
  VMD_POS: [0.038, 0.81, -0.004]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.038, 0.81, 0.004]
  最大差異: 0.810000

骨骼名稱: bone0382_Ches
  VMD_POS: [0.023, 0.77, 0.004]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.023, 0.77, 0.004]
  最大差異: 0.770000

骨骼名稱: bone0488_Butt
  VMD_POS: [-0.097, 0.742, -0.515]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.097, 0.742, 0.515]
  最大差異: 0.742000

骨骼名稱: bone0453_Butt
  VMD_POS: [-0.63, 0.396, -0.555]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.63, 0.396, 0.555]
  最大差異: 0.630000

骨骼名稱: BSkirt_6_2
  VMD_POS: [-0.232, 0.01, 0.545]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.232, 0.01, 0.545]
  最大差異: 0.545000

骨骼名稱: BSkirt_6_1
  VMD_POS: [-0.247, -0.007, 0.537]
  FINAL_POS: [0.0, 0.0, 0.0]
  各軸差異: [0.247, 0.